# Small-island edge cases

Handles small-island nations whose indicators are missing or reported
differently in the source data.

In [ ]:
import pandas as pd
from scipy.spatial import cKDTree
import numpy as np

In [ ]:
df = pd.read_csv('df.tsv', sep='\t', encoding='utf-8')

## Identify missing small islands

Several small-island nations are absent from the merged data; the cells below locate and fill them.

## Check for missing subnational geography

## Fill missing values

Where possible, fill a language's subnational values from other languages sharing its `admin_1_code`, assigning `admin_1_code` by geographical proximity within the same country.

In [ ]:
# Ensure lat/lon are floats
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# Separate rows with and without admin_1_code
missing_mask = df['admin_1_code'].isna() | (df['admin_1_code'].astype(str).str.strip() == '')
df_missing = df[missing_mask].copy()
df_known = df[~missing_mask].copy()

# Function to find nearest neighbor by Haversine (approx using KDTree over lat/lon)
def fill_missing_admin(df_missing, df_known):
    filled_codes = []

    for idx, row in df_missing.iterrows():
        country_code = row['country_iso_code']
        lat, lon = row['latitude'], row['longitude']

        # Filter known values in the same country
        same_country = df_known[df_known['country_iso_code'] == country_code].dropna(
            subset=['latitude', 'longitude', 'admin_1_code'])

        if same_country.empty or pd.isna(lat) or pd.isna(lon):
            filled_codes.append(np.nan)
            continue

        # Build KDTree for fast lookup
        coords = same_country[['latitude', 'longitude']].values
        tree = cKDTree(coords)
        dist, idx_nearest = tree.query([[lat, lon]], k=1)
        nearest_code = same_country.iloc[idx_nearest[0]]['admin_1_code']
        filled_codes.append(nearest_code)

    df_missing['admin_1_code'] = filled_codes
    return df_missing

# Fill in missing values
df_filled = fill_missing_admin(df_missing, df_known)

# Combine back into original dataframe
df.loc[df_filled.index, 'admin_1_code'] = df_filled['admin_1_code']

In [ ]:
columns_to_fill = [
    'hdi',
    'gdp',
    'Cybersecurity legislation/regulation exist',
    'households with phones',
    'households with home internet',
    'individuals_using_internet',
    'avg_d_kbps',
    'avg_u_kbps',
    'avg_lat_ms',
    '% Stem Grads',
    'literacy rate',
    'educational_attainment_score',
    'RnD % of GDP'
]

In [ ]:
for idx, row in df_filled.iterrows():
    admin_code = row['admin_1_code']

    # Filter rows with the same admin_1_code, excluding current row
    same_admin_rows = df[(df['admin_1_code'] == admin_code) & (df.index != idx)]

    if not same_admin_rows.empty:
        for col in columns_to_fill:
            if col in df.columns:
                # Filter out zeros and NaNs, then get the first value if exists
                non_zero_values = same_admin_rows[col][(same_admin_rows[col] != 0) & (~same_admin_rows[col].isna())]
                if not non_zero_values.empty:
                    df_filled.at[idx, col] = non_zero_values.iloc[0]

In [ ]:
df.loc[df_filled.index, columns_to_fill] = df_filled[columns_to_fill]

In [ ]:
empty_admin_1_code_rows = df[df['admin_1_code'].isna() | (df['admin_1_code'].astype(str).str.strip() == '')]

unique_country_codes = empty_admin_1_code_rows['country_iso_code'].dropna().unique()

print(list(unique_country_codes))